### Tóm tắt cách vận hành:
- Lần chạy 1: Đặt OFFSET = 0, BATCH_SIZE = 35000. Bấm Save & Run All, tắt máy.
- Lần chạy 2 (sau khi lần 1 xong hoặc chạy song song trên Notebook khác): Đặt OFFSET = 35000, BATCH_SIZE = 35000.

Bộ đếm Time Guard (10.8 tiếng) đảm bảo dù mạng có chậm đến mấy thì máy chủ Kaggle luôn dừng kịp để đóng gói zip ra tab Output, không bao giờ bị lỗi quá 12h!

Mỗi lần chạy xong sẽ tạo ra một file zip riêng biệt: corpus_b_small_offset_0.zip, corpus_b_small_offset_35000.zip... sẵn sàng trong tab Output để bạn tải về bất kỳ lúc nào!

In [ ]:
# ==============================================================================
# R2AI 2026 - KAGGLE CRAWLER: 49 DOMAIN NHỎ + TIME GUARD + PHÂN LÔ THEO BATCH
# ==============================================================================

# 1. CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT
!pip install -q trafilatura zstandard pyarrow datasets aiohttp protego


In [ ]:
import os
import sys
import time
import json
import asyncio
import hashlib
import unicodedata
from datetime import datetime, timezone, timedelta
from urllib.parse import urlsplit
from typing import Dict, Any, List

import aiohttp
import trafilatura
import zstandard as zstd
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset

# ==============================================================================
# 2. CẤU HÌNH THÔNG SỐ BATCH VÀ TIME GUARD
# ==============================================================================
OWNER = "b"

# Thay đổi OFFSET sau mỗi lần chạy (Lần 1: 0, Lần 2: 35000, Lần 3: 70000...)
OFFSET = 0                        
BATCH_SIZE = 35000                # Số URL cào trong 1 phiên
MAX_RUNTIME_HOURS = 10.8          # Bộ đếm an toàn: tự dừng trước 10.8 tiếng để kịp nén file zip

START_TIME = time.time()
MAX_RUNTIME_SECONDS = MAX_RUNTIME_HOURS * 3600

KAGGLE_OUT = "/kaggle/working"
DIR_DOCS = f"{KAGGLE_OUT}/corpus/viz/docs/{OWNER}"
DIR_LOGS = f"{KAGGLE_OUT}/corpus/viz/crawl_log/{OWNER}"
DIR_MANIFEST = f"{KAGGLE_OUT}/corpus/viz/manifest/{OWNER}"

for d in [DIR_DOCS, DIR_LOGS, DIR_MANIFEST]:
    os.makedirs(d, exist_ok=True)

# ==============================================================================
# 3. DANH SÁCH 49-51 DOMAIN NHỎ CỦA MEMBER B (ĐÃ LOẠI 2 DOMAIN KHỦNG)
# ==============================================================================
DOMAINS_B_SMALL = {
    "a-hospital.com", "suckhoecongdongonline.vn", "suckhoedoisong.vn", "thanhnien.vn",
    "wujue.com", "baby.39.net", "vinmec.com", "medlatec.vn", "jbk.39.net",
    "giadinhonline.vn", "baonghean.vn", "care.39.net", "baodanang.vn", "fitness.39.net",
    "hellobacsi.com", "food.39.net", "fk.39.net", "cancer.39.net", "jb39.com",
    "vov2.vov.vn", "phuyen.baodaklak.vn", "heart.39.net", "suckhoeviet.org.vn",
    "pmc-ecm-healthblog.beta.pharmacity.io", "nhandan.vn", "gan.39.net", "wei.39.net",
    "baidianfeng.familydoctor.com.cn", "vietnamplus.vn", "shen.39.net", "tienphong.vn",
    "thaythuocvietnam.vn", "bachmai.gov.vn", "dantri.com.vn", "gk.39.net",
    "baolangson.vn", "gc.39.net", "baophutho.vn", "benhviennhitrunguong.gov.vn",
    "sj.39.net", "baothanhhoa.vn", "pharmacity.vn", "familydoctor.cn",
    "qy.familydoctor.com.cn", "v.familydoctor.com.cn", "ask.familydoctor.com.cn",
    "mega.vietnamplus.vn", "yanglao.familydoctor.com.cn", "fk.99.com.cn",
    "special.vietnamplus.vn", "ypk.familydoctor.com.cn"
}

# ==============================================================================
# 4. TẢI VÀ LỌC CẮT PHÂN ĐOẠN (SLICING) TỪ HUGGING FACE
# ==============================================================================
print("[*] Đang kết nối tới Hugging Face: AIGuruTinix/ViBioMIR...")
hf_dataset = load_dataset("AIGuruTinix/ViBioMIR", "corpus", split="train")

def clean_hostname(url_str):
    try:
        h = urlsplit(url_str).hostname or ""
        h = h.lower().rstrip(".")
        return h[4:] if h.startswith("www.") else h
    except:
        return ""

print("[*] Đang lọc các URL thuộc nhóm domain nhỏ...")
all_small_urls = []
for row in hf_dataset:
    domain = clean_hostname(row["url"])
    if domain in DOMAINS_B_SMALL:
        all_small_urls.append({"id": int(row["id"]), "url": row["url"], "domain": domain})

total_available = len(all_small_urls)
print(f"[✓] Tìm thấy tổng cộng: {total_available:,} URLs từ các domain nhỏ.")

# CẮT PHÂN LÔ THEO OFFSET VÀ BATCH_SIZE
target_batch = all_small_urls[OFFSET : OFFSET + BATCH_SIZE]
print(f"[*] Phiên này sẽ cào: Index #{OFFSET:,} -> #{OFFSET + len(target_batch):,} (Tổng: {len(target_batch):,} URLs)")

# ==============================================================================
# 5. CHUẨN HÓA VĂN BẢN (QUY TẮC N1: UNICODE NFC)
# ==============================================================================
def normalize_text(raw_text: str) -> str:
    if not raw_text:
        return ""
    text = unicodedata.normalize("NFC", raw_text).replace("\r\n", "\n").replace("\r", "\n")
    text = "".join(ch for ch in text if ch in ("\n", "\t") or not unicodedata.category(ch).startswith("C"))
    lines = [" ".join(line.split()) for line in text.split("\n")]
    res = "\n".join(lines)
    while "\n\n\n" in res:
        res = res.replace("\n\n\n", "\n\n")
    return res.strip()

# ==============================================================================
# 6. BỘ GHI SHARD AN TOÀN (ATOMIC ZSTD SHARD WRITER)
# ==============================================================================
class ShardWriter:
    def __init__(self, start_seq=1, shard_size=20000):
        self.shard_size = shard_size
        self.seq = start_seq
        self.count = 0
        self.cctx = zstd.ZstdCompressor(level=3)
        self._new_shard()

    def _new_shard(self):
        self.shard_name = f"viz-{OWNER}-small-{self.seq:05d}.jsonl.zst"
        self.final_path = os.path.join(DIR_DOCS, self.shard_name)
        self.temp_path = self.final_path + ".tmp"
        self.f_out = open(self.temp_path, "wb")
        self.writer = self.cctx.stream_writer(self.f_out)
        self.count = 0

    def write(self, doc):
        line = (json.dumps(doc, ensure_ascii=False) + "\n").encode("utf-8")
        self.writer.write(line)
        self.count += 1
        if self.count >= self.shard_size:
            self.close()
            self.seq += 1
            self._new_shard()

    def close(self):
        if self.writer:
            self.writer.close()
            self.f_out.close()
        if self.count > 0 and os.path.exists(self.temp_path):
            os.rename(self.temp_path, self.final_path)
            hasher = hashlib.sha256()
            with open(self.final_path, "rb") as f:
                while chunk := f.read(65536):
                    hasher.update(chunk)
            manifest = {
                "shard": self.shard_name,
                "owner": OWNER,
                "schema_version": "1.0",
                "norm_version": "n1",
                "n_docs": self.count,
                "sha256": hasher.hexdigest(),
                "created_at": datetime.now(timezone(timedelta(hours=7))).isoformat()
            }
            with open(os.path.join(DIR_MANIFEST, f"{self.shard_name}.manifest.json"), "w") as mf:
                json.dump(manifest, mf, indent=2)
            print(f"\n[✓ ĐÃ ĐÓNG SHARD] {self.shard_name} ({self.count:,} docs)")

# ==============================================================================
# 7. ASYNC CRAWLER KÈM BỘ KHÓA DOMAIN & TIME GUARD
# ==============================================================================
async def crawl_batch():
    start_seq = (OFFSET // 20000) + 1
    shard_writer = ShardWriter(start_seq=start_seq, shard_size=20000)
    logs = []
    total = len(target_batch)
    completed = 0
    stop_signal = False

    headers = {"User-Agent": "R2AI2026CorpusResearch/1.0 (Kaggle respectful multi-domain crawler)"}
    connector = aiohttp.TCPConnector(limit=32)
    timeout = aiohttp.ClientTimeout(total=20)

    # Mỗi domain trong 49 domain có 1 khóa và giãn cách 1 giây riêng biệt
    domain_locks = {d: asyncio.Lock() for d in DOMAINS_B_SMALL}
    next_call = {d: 0.0 for d in DOMAINS_B_SMALL}
    semaphore = asyncio.Semaphore(32)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout, headers=headers) as session:
        
        async def fetch_url(item):
            nonlocal completed, stop_signal
            
            # KIỂM TRA TIME GUARD TRƯỚC MỖI REQUEST
            if (time.time() - START_TIME) >= MAX_RUNTIME_SECONDS:
                stop_signal = True
                return

            d = item["domain"]
            lock = domain_locks.get(d)
            status = "ok"
            http_status = None
            err = None
            final_text = ""
            title = ""

            async with semaphore:
                if lock:
                    async with lock:
                        wait = next_call[d] - time.time()
                        if wait > 0:
                            await asyncio.sleep(wait)
                        next_call[d] = time.time() + 1.0  # Giãn cách 1 giây cho cùng 1 domain

                        try:
                            async with session.get(item["url"], allow_redirects=True) as resp:
                                http_status = resp.status
                                if resp.status == 200:
                                    html = await resp.text(errors="replace")
                                    extracted = trafilatura.extract(html, include_comments=False, include_tables=True)
                                    if not extracted or len(extracted.strip()) == 0:
                                        status = "empty"
                                    else:
                                        final_text = normalize_text(extracted)
                                        meta = trafilatura.extract_metadata(html)
                                        title = normalize_text(meta.title or "") if meta else ""
                                        if len(final_text) < 100:
                                            status = "too_short"
                                else:
                                    status = "http_error"
                                    err = f"HTTP_{http_status}"
                        except asyncio.TimeoutError:
                            status = "timeout"
                            err = "Timeout"
                        except Exception as e:
                            status = "http_error"
                            err = str(e)[:60]

            # Ghi Document vào shard nếu hợp lệ
            if status in ("ok", "too_short") and final_text:
                doc = {
                    "schema_version": "1.0",
                    "uid": f"src:{item['id']}",
                    "doc_id": str(item["id"]),
                    "lang": "zh" if ".cn" in d else "vi",
                    "source": "btc_url",
                    "url": item["url"],
                    "domain": d,
                    "title": title,
                    "text": final_text,
                    "norm_version": "n1",
                    "passages": [{"type": "body", "start": 0, "end": len(final_text)}],
                    "text_len": len(final_text),
                    "content_hash": f"sha1:{hashlib.sha1(final_text.encode('utf-8')).hexdigest()}",
                    "dup_of": None,
                    "fetch": {
                        "status": status,
                        "http_status": http_status,
                        "fetched_at": datetime.now(timezone(timedelta(hours=7))).isoformat()
                    },
                    "owner": OWNER
                }
                shard_writer.write(doc)

            logs.append({
                "id": item["id"], "url": item["url"], "domain": d,
                "status": status, "http_status": http_status, "error": err
            })

            completed += 1
            if completed % 100 == 0:
                elapsed = time.time() - START_TIME
                rate = completed / elapsed if elapsed > 0 else 0
                eta_m = (total - completed) / rate / 60 if rate > 0 else 0
                print(f"\rTiến độ: {completed:,}/{total:,} ({completed/total*100:.1f}%) | {rate:.1f} URL/s | ETA: {eta_m:.1f} phút", end="", flush=True)

        # Chạy đồng thời qua danh sách batch
        tasks = [fetch_url(item) for item in target_batch]
        await asyncio.gather(*tasks)

    shard_writer.close()
    
    # Lưu nhật ký crawl Parquet
    if logs:
        log_file = f"{DIR_LOGS}/log-{OWNER}-offset_{OFFSET}.parquet"
        pq.write_table(pa.Table.from_pylist(logs), log_file, compression="zstd")
        print(f"\n[✓] Đã lưu file log: {log_file}")

# KHỞI ĐỘNG CRAWLER
print("[*] Bắt đầu cào dữ liệu...")
asyncio.run(crawl_batch())

# ==============================================================================
# 8. TỰ ĐỘNG NÉN FILE ZIP THEO ĐÚNG TÊN BATCH VÀO OUTPUT
# ==============================================================================
zip_filename = f"corpus_b_small_offset_{OFFSET}.zip"
print(f"\n[*] Đang nén kết quả vào {KAGGLE_OUT}/{zip_filename}...")
!cd /kaggle/working && zip -r -q {zip_filename} corpus/
print(f"[✓] THÀNH CÔNG RỰC RỠ! Bạn có thể tải file '{zip_filename}' trong tab Output.")